# Chapter 15: MLOps & Model Deployment
## Notebook 02 — Pipelines & CI/CD

Now that we can package and serve, we tackle **reproducibility** and **automation**: how do we make model training deterministic, track experiments, manage versions in a registry, and gate deploys with CI?

### What you'll learn

| Topic | Section |
|-------|--------|
| sklearn `Pipeline` and reproducibility (seeds, lockfiles) | §1–2 |
| Experiment tracking: `mlflow` with a JSON fallback | §3 |
| File-backed model registry: stages and promotion gates | §4 |
| CI/CD for ML: lint → test → train → eval → register → deploy | §5 |
| The data / code / model versioning triplet | §6 |
| A reproducibility checklist | §7 |

**Estimated time:** 2.5 hours

---
*Generated by Berta AI*

---
## 1. Pipelines: One Object, One Train Path

Two-step modeling (preprocess script + model script) is a classic source of training/serving skew: a transformation gets applied at train time but not at serve time, or vice versa. The cure is to keep **everything inside one fitted pipeline object** that travels through serialization.

A scikit-learn `Pipeline` does exactly this: it composes transformers and a final estimator and exposes a single `.fit / .predict / .score` interface.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import config
from registry import ModelRegistry

print('chapter root:', config.chapter_root())

In [ ]:
# Build a pipeline. Notice: the ONLY scaling that ever happens is what's inside.
def build_pipeline(C: float = 1.0, seed: int = 42) -> Pipeline:
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=C, max_iter=200, random_state=seed)),
    ])

rng = np.random.default_rng(0)
n = 600
X = rng.normal(size=(n, 2))
y = ((X[:, 0] + 0.5 * X[:, 1]) > 0).astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)

pipe = build_pipeline()
pipe.fit(Xtr, ytr)

metrics = {
    'accuracy': float(accuracy_score(yte, pipe.predict(Xte))),
    'f1': float(f1_score(yte, pipe.predict(Xte))),
}
print('eval metrics:', {k: round(v, 4) for k, v in metrics.items()})

---
## 2. Reproducibility: Seeds, Lockfiles, and the Same Numbers Twice

A reproducible run produces **bitwise identical** outputs given the same code, data, and dependencies. In practice you control three knobs:

1. **Random state** — set seeds for `numpy`, `random`, your framework, and *every estimator* that takes a `random_state`.
2. **Pinned dependencies** — `requirements.txt` with `==` pins, ideally a `pip-compile` lockfile or `uv.lock`.
3. **Pinned data** — record the dataset hash; tools like DVC or LakeFS do this for you.

The acid test: train the same pipeline twice and compare predictions exactly.

In [ ]:
# Determinism check: same seed -> same predictions
p1 = build_pipeline(seed=42).fit(Xtr, ytr)
p2 = build_pipeline(seed=42).fit(Xtr, ytr)
same = (p1.predict(Xte) == p2.predict(Xte)).all()
print('seed=42 -> identical predictions:', bool(same))

# Different seed
p3 = build_pipeline(seed=123).fit(Xtr, ytr)
diff = (p1.predict(Xte) != p3.predict(Xte)).sum()
print('seed=123 differs in', int(diff), 'of', len(yte), 'predictions')

In [ ]:
# Hash the data and the code for the run record.
import hashlib

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()[:16]

data_hash = sha256_bytes(np.ascontiguousarray(Xtr).tobytes() + np.ascontiguousarray(ytr).tobytes())
code_hash = sha256_bytes(open(__file__).read().encode() if '__file__' in dir() else b'notebook')
print('data_hash:', data_hash)
print('code_hash:', code_hash, '(would be a git commit SHA in CI)')

---
## 3. Experiment Tracking

A single trained model is one row in a long table of experiments. **Experiment tracking** records, for each run: hyperparameters, metrics, the artifact, the code commit, and the data version. MLflow is the canonical Python tool; if it isn't installed we fall back to a JSON tracker.

In [ ]:
class JsonTracker:
    """Tiny stand-in for an experiment tracker. One JSON file per experiment."""
    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        if not self.path.exists():
            self.path.write_text('[]')
    def log_run(self, params, metrics, tags=None):
        runs = json.loads(self.path.read_text())
        run = {
            'run_id': f'run_{len(runs):04d}',
            'timestamp': time.time(),
            'params': params, 'metrics': metrics, 'tags': dict(tags or {}),
        }
        runs.append(run)
        self.path.write_text(json.dumps(runs, indent=2))
        return run

# Try MLflow; fall back if it isn't installed
try:
    import mlflow  # type: ignore
    USE_MLFLOW = True
    print('mlflow available — would call mlflow.start_run() here')
except ImportError:
    USE_MLFLOW = False
    print('mlflow not installed — falling back to JsonTracker (pip install mlflow to upgrade)')

tracker = JsonTracker('../results/experiments.json')
runs = []
for C in [0.1, 1.0, 10.0]:
    p = build_pipeline(C=C).fit(Xtr, ytr)
    m = {
        'accuracy': float(accuracy_score(yte, p.predict(Xte))),
        'f1': float(f1_score(yte, p.predict(Xte))),
    }
    run = tracker.log_run({'C': C, 'seed': 42, 'model': 'logreg'}, m, tags={'data_hash': data_hash})
    runs.append((C, m))
    print(f'  C={C:>5} -> acc={m["accuracy"]:.3f}, f1={m["f1"]:.3f}')
print('\nlogged', len(runs), 'runs to', tracker.path)

**Pick the best run** (by some metric) and promote that artifact to the registry. We'll wire that up next.

---
## 4. The Model Registry

A registry is the source of truth for "what's deployed." Every artifact has a **stage**:

```
None  →  Staging  →  Production  →  Archived
```

- **None** — just registered, not yet promoted.
- **Staging** — passed offline gates; running shadow / canary traffic.
- **Production** — currently serving real users.
- **Archived** — retired. Kept for audit and rollback.

Our `ModelRegistry` (in `scripts/registry.py`) is file-backed: a single JSON index with on-disk artifacts. The API mirrors MLflow's so the patterns transfer.

In [ ]:
# Save the best pipeline to a temp artifact, then register it.
best_C = max(runs, key=lambda r: r[1]['f1'])[0]
best_pipe = build_pipeline(C=best_C).fit(Xtr, ytr)
best_metrics = {
    'accuracy': float(accuracy_score(yte, best_pipe.predict(Xte))),
    'f1': float(f1_score(yte, best_pipe.predict(Xte))),
}

art_dir = Path('../models'); art_dir.mkdir(exist_ok=True)
artifact = art_dir / f'demo_v0.1.0.joblib'
joblib.dump(best_pipe, artifact)
print('artifact:', artifact, '(size', artifact.stat().st_size, 'bytes)')

reg = ModelRegistry('../registry')
entry = reg.register(
    model_name='demo-classifier',
    version='0.1.0',
    artifact_src=artifact,
    framework='sklearn',
    metrics=best_metrics,
    tags={'data_hash': data_hash, 'best_C': str(best_C)},
)
print('\nregistered entry:')
print(json.dumps(entry.to_dict(), indent=2, default=str))

In [ ]:
# Apply the promotion gate. Refuse to promote if the model fails offline thresholds.
GATES = {
    'min_accuracy': 0.85,
    'min_f1': 0.80,
}

def passes_gates(metrics: dict, gates: dict) -> bool:
    return (metrics['accuracy'] >= gates['min_accuracy']
            and metrics['f1'] >= gates['min_f1'])

if passes_gates(best_metrics, GATES):
    reg.transition_stage('demo-classifier', '0.1.0', 'Staging')
    reg.transition_stage('demo-classifier', '0.1.0', 'Production')
    print('promoted to Production')
else:
    print('FAILED gates — not promoted. metrics:', best_metrics)

prod = reg.get_production('demo-classifier')
print('\ncurrent Production version:', prod.version if prod else None)

In [ ]:
# Promoting a NEW version auto-archives the previous Production version.
artifact_v2 = art_dir / 'demo_v0.2.0.joblib'
joblib.dump(best_pipe, artifact_v2)  # same model — just demonstrating the flow
reg.register('demo-classifier', '0.2.0', artifact_v2, metrics=best_metrics)
reg.transition_stage('demo-classifier', '0.2.0', 'Production')

for e in reg.list_models('demo-classifier'):
    print(f'  {e.version:6}  stage={e.stage}')

---
## 5. CI/CD for ML

A typical ML pull request should run, in order:

```
lint  →  unit tests  →  train (small)  →  eval gates  →  register  →  deploy gate (manual)
```

Below is a sample GitHub Actions workflow. It runs every push, trains on a small slice, evaluates against thresholds, and only registers + deploys when all gates pass.

In [ ]:
WORKFLOW_YAML = '''
name: train-and-deploy

on:
  push:
    branches: [ main ]
  pull_request:

jobs:
  ci:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip
      - run: pip install -r requirements.txt
      - name: Lint
        run: |
          python -m pip install ruff
          ruff check scripts
      - name: Unit tests
        run: python -m pytest tests/ -q
      - name: Train (smoke)
        run: python scripts/train.py --smoke
      - name: Eval gates
        run: |
          python -c "import json; m=json.load(open(\"results/metrics.json\")); \\
                     assert m[\"accuracy\"] >= 0.80 and m[\"f1\"] >= 0.75, m"
      - name: Register
        if: github.ref == 'refs/heads/main'
        run: python scripts/register.py --stage Staging
      - name: Deploy (manual approval)
        if: github.ref == 'refs/heads/main'
        environment:
          name: production
          url: https://api.example.com
        run: ./scripts/deploy.sh
'''.strip()
print(WORKFLOW_YAML[:600], '...\n[truncated]')

**Why these gates matter:**

- **Lint + unit tests** catch the easy stuff before any compute is spent.
- **Smoke train** catches data-loading bugs that only happen end-to-end.
- **Eval gates** are the model-quality contract: accuracy / F1 must beat the current Production model (or a fixed floor).
- **Manual approval** for the deploy step is your last line of defense for production-only side effects.

---
## 6. Versioning the Triplet: Data, Code, Model

A "version" of an ML system is really three things:

| Layer | Tool | What's stored |
|-------|------|---------------|
| **Code** | git | Source SHA |
| **Data** | DVC, LakeFS, S3+hash | Dataset version pointer |
| **Model** | MLflow registry, our `ModelRegistry` | Artifact + metrics + lineage |

A registry entry should reference *both* a `code_sha` and a `data_hash` so any deployment is fully reproducible from its three coordinates.

In [ ]:
# Show the lineage we recorded on our entry.
prod = reg.get_production('demo-classifier')
print('production lineage:')
print(json.dumps(prod.to_dict(), indent=2, default=str))

---
## 7. Reproducibility Checklist

Use this before merging any model PR:

- [ ] Random seeds set for NumPy, framework, and every estimator
- [ ] Dependencies pinned in `requirements.txt` (or lockfile)
- [ ] Data version recorded (hash, snapshot id, or DVC pointer)
- [ ] Code version recorded (git SHA)
- [ ] All preprocessing inside a single fitted pipeline (no out-of-band steps)
- [ ] Train/eval split is deterministic and recorded
- [ ] Metrics logged to a tracker, not just printed
- [ ] Artifact registered with metrics + lineage tags
- [ ] Promotion gates encoded in CI, not enforced by humans
- [ ] Same code path used to train and to evaluate before promotion

---
## 8. Key Takeaways

- A **single fitted pipeline** kills training/serving skew at the root.
- **Reproducibility** = seeds + pinned deps + pinned data + recorded code SHA.
- **Experiment trackers** turn ad-hoc notebook runs into queryable history.
- A **registry** holds the lifecycle: None → Staging → Production → Archived, with at most one Production version.
- **CI/CD** makes promotion gates non-negotiable; humans don't override green/red.
- **Versioning is a triplet**: code, data, model. Record all three on every registered artifact.

Next: **Notebook 03** — drift, A/B and canary deploys, observability, scaling, and the capstone.

---
*Generated by Berta AI*